# 第 56 课｜pytest 量化单元测试

本 Notebook 对应「365 天量化金融」第 56 课。课程方法来自本地完整视频、课件截图和本地 Whisper 口播整理；这里把它重建为可运行、可验证的量化工程教学实现。

- 图文口播：`/Users/mojiawen/Documents/学习资料归档/_暂存/pam-video-captures/365-days-quant-finance/56_youtube-gKF2PotbNDU/index.html`
- 环境：`pkm-quant`
- 数据：可手算的小型合成价格、信号和成本
- 状态：可运行学习稿；不代表生产交易系统

## 本节量化问题

单元测试把收益公式、信号滞后、手续费和会计规则变成自动护栏。本节先在 Notebook 展示 AAA 模式，再生成一个真正由 pytest 执行的测试文件。

In [1]:
from pathlib import Path
import json
import os
import platform
import sys
import tempfile
import timeit

# Matplotlib 首次运行需要写字体缓存；放到可写临时目录，避免污染用户目录。
MPL_CACHE_DIR = Path(tempfile.gettempdir()) / "pkm-quant-matplotlib"
MPL_CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(MPL_CACHE_DIR))

import matplotlib.pyplot as plt
from matplotlib import font_manager
import numpy as np
import pandas as pd
import seaborn as sns

# 从本机字体中选择支持中文的字体，避免标题显示成方框。
preferred_chinese_fonts = [
    "PingFang SC",
    "Arial Unicode MS",
    "Hiragino Sans GB",
    "Heiti SC",
]
available_font_names = {font.name for font in font_manager.fontManager.ttflist}
chinese_font = next(
    (name for name in preferred_chinese_fonts if name in available_font_names),
    "DejaVu Sans",
)
plt.rcParams["font.sans-serif"] = [chinese_font, "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

def find_project_root():
    cwd = Path.cwd().resolve()
    candidates = [
        cwd,
        cwd / "实验" / "p2-python-toolchain",
        cwd / "p2-python-toolchain",
        *cwd.parents,
    ]
    for candidate in candidates:
        if (candidate / "课程辅助设计.md").exists():
            return candidate
    raise FileNotFoundError("请从 p2-python-toolchain 或量化交易项目目录启动 JupyterLab")

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data" / "fixtures"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)
RANDOM_SEED = 365

print("Python:", sys.version.split()[0])
print("解释器:", sys.executable)
print("项目:", PROJECT_ROOT)
print("绘图字体:", chinese_font)

Python: 3.11.15
解释器: /opt/homebrew/Caskroom/miniforge/base/envs/pkm-quant/bin/python
项目: /Users/mojiawen/Documents/学习资料归档/_暂存/personal_knowledge_manager/量化交易/实验/p2-python-toolchain
绘图字体: PingFang SC


In [2]:
# pytest 已进入统一环境；subprocess 用同一解释器运行测试。
import pytest
import subprocess
import textwrap
print("pytest:", pytest.__version__)

pytest: 9.1.1


## 1. AAA：Arrange、Act、Assert

下面价格从 100 到 110 再到 99，日收益应为 10% 和 -10%。期望值来自手算，不来自待测函数自身。

In [3]:
# Arrange：准备价格；Act：计算收益；Assert：与手算期望比较。
def simple_returns(prices):
    prices = np.asarray(prices, dtype=float)
    if len(prices) < 2:
        return np.array([], dtype=float)
    return prices[1:] / prices[:-1] - 1

prices = np.array([100.0, 110.0, 99.0])
actual = simple_returns(prices)
expected = np.array([0.10, -0.10])
np.testing.assert_allclose(actual, expected, rtol=1e-12, atol=1e-12)
print("实际:", actual, "期望:", expected)

实际: [ 0.1 -0.1] 期望: [ 0.1 -0.1]


## 2. 参数化测试覆盖正常值、空样本和零分母

价格为零不应悄悄得到无穷大，因此稳健函数显式拒绝非正价格。`pytest.raises` 验证预期异常。

In [4]:
# 定义带输入契约的收益函数，并在 Notebook 中直接检查边界。
def validated_returns(prices):
    values = np.asarray(prices, dtype=float)
    if np.any(values <= 0):
        raise ValueError("prices must be positive")
    return simple_returns(values)

assert validated_returns([100.0]).size == 0
with pytest.raises(ValueError, match="positive"):
    validated_returns([100.0, 0.0])

## 3. 生成并实际运行 pytest 文件

测试覆盖：

- 可手算收益；
- 参数化边界；
- 持仓必须使用上一期信号；
- 成本后终值不高于无成本终值。

In [5]:
# 写入独立测试文件，并用当前 pkm-quant 解释器运行。
test_source = textwrap.dedent('''
import numpy as np
import pytest

def simple_returns(prices):
    values = np.asarray(prices, dtype=float)
    if np.any(values <= 0):
        raise ValueError("prices must be positive")
    return values[1:] / values[:-1] - 1

@pytest.mark.parametrize("prices, expected", [
    ([100.0, 110.0], [0.10]),
    ([100.0], []),
    ([100.0, 90.0, 99.0], [-0.10, 0.10]),
])
def test_returns(prices, expected):
    np.testing.assert_allclose(simple_returns(prices), expected, rtol=1e-12, atol=1e-12)

def test_zero_price_rejected():
    with pytest.raises(ValueError):
        simple_returns([100.0, 0.0])

def test_signal_is_lagged_and_cost_is_monotone():
    signal = np.array([1.0, 0.0, 1.0])
    position = np.r_[0.0, signal[:-1]]
    returns = np.array([0.01, -0.02, 0.03])
    gross = position * returns
    turnover = np.abs(np.diff(np.r_[0.0, position]))
    net = gross - 0.001 * turnover
    assert position[1] == signal[0]
    assert np.prod(1 + net) <= np.prod(1 + gross)
''')
test_path = OUTPUT_DIR / "56_test_quant_core.py"
test_path.write_text(test_source, encoding="utf-8")
completed = subprocess.run(
    [sys.executable, "-m", "pytest", "-q", str(test_path)],
    capture_output=True, text=True, check=False,
)
print(completed.stdout)
if completed.stderr:
    print(completed.stderr)
assert completed.returncode == 0

.....                                                                    [100%]
5 passed in 0.04s



## 方法边界与复用结论


- 测试通过只保护已经声明的行为；未覆盖的错误仍然存在。
- 期望值应来自定义、手算或独立参考实现，不能把当前输出复制成“正确答案”。
- 网络接口契约测试和纯计算单元测试应分开，避免外部不稳定污染核心测试。


本课首先保护数据正确性、时间一致性和可复现性，再讨论速度或自动化。局部代码通过不代表整套交易系统满足权限、监控、灾备和实盘延迟要求。